Install dependencies

In [1]:
# Install Hugging Face libraries
!pip install -q transformers[torch] datasets sentencepiece accelerate

Use processed dataset and merge into one training set

In [2]:
import pandas as pd
from datasets import Dataset

# Load the two CSV files
df1 = pd.read_csv('informal_formal.csv')
df2 = pd.read_csv('seed_pairs.csv')

# Combine them into a single dataframe
# We drop duplicates in case the seed pairs are already in the informal_formal file
df_combined = pd.concat([df1, df2]).drop_duplicates().reset_index(drop=True)

print(f"Total training examples: {len(df_combined)}")

# T5 is a multi-task model, so we add a prefix to the input
# so the model knows we are doing style transfer.
df_combined['informal'] = "transfer informal to formal: " + df_combined['informal']

# Convert to Hugging Face Dataset format
dataset = Dataset.from_pandas(df_combined)

# Split into Training (90%) and Validation (10%)
dataset = dataset.train_test_split(test_size=0.1)

Total training examples: 362


Tokenization using T5tokenizer

In [3]:
from transformers import T5Tokenizer

model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    # Tokenize informal inputs
    inputs = tokenizer(examples["informal"], max_length=128, truncation=True, padding="max_length")
    # Tokenize formal targets
    targets = tokenizer(examples["formal"], max_length=128, truncation=True, padding="max_length")

    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/325 [00:00<?, ? examples/s]

Map:   0%|          | 0/37 [00:00<?, ? examples/s]

Using trainer API for training

In [ ]:
from transformers import T5ForConditionalGeneration, TrainingArguments, Trainer

model = T5ForConditionalGeneration.from_pretrained(model_name)

training_args = TrainingArguments(
    output_dir="./formal_transformer_checkpoints",
    eval_strategy="epoch",        # Changed from evaluation_strategy to eval_strategy
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    save_total_limit=2,
    logging_steps=50,
    push_to_hub=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
)

trainer.train()

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


Inference and testing

In [ ]:
def formalize(text):
    # Prepare the input with the task prefix
    input_text = "transfer informal to formal: " + text

    # Encode and generate
    inputs = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)
    outputs = model.generate(
        inputs,
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

    # Decode back to readable text
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Try it out
test_sentence = "Just got a sick new phone, it's lit."
print(f"Informal: {test_sentence}")
print(f"Formal:   {formalize(test_sentence)}")